# Session 7 — Quantum Information Theory

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S5 gave us the classical information toolbox (entropy, KL, mutual information).
S6 turned the simplex into a *geometry*. S7 lifts both moves to **density matrices**.
Two effects appear that **have no classical analogue**:

1. **Quantum mutual information can be twice the classical maximum** (2 bits for a Bell pair).
2. **Conditional entropy can be negative** ($S(A|B) = -1$ bit for a Bell pair).

Both come from entanglement. We close M2 — the spine — with the dictionary's information
row fully filled in.

## 0. What you'll be able to do

- Compute **Umegaki relative entropy** $S(\rho \| \sigma) = \mathrm{tr}\,\rho(\log\rho - \log\sigma)$ and see it distinguish $|+\rangle$ from $I/2$ — even though their measurement statistics in the $Z$ basis are *identical*.
- Compute the **quantum mutual information** $I(A{:}B)$ for a Bell pair and watch it reach **2 bits** (double the classical bound).
- Compute the **quantum conditional entropy** $S(A|B)$ and find it **negative** for entangled states.
- Sweep a Werner state $\rho(p) = (1-p)|\mathrm{Bell}\rangle\langle\mathrm{Bell}| + p\,I/4$ and watch $I(A{:}B)$ and $S(A|B)$ cross the classical bounds together.
- Compute the **Bures distance** — the quantum lift of the Fisher–Rao distance of S6.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.infotheory.quantum import (
    quantum_relative_entropy,
    quantum_mutual_information,
    quantum_conditional_entropy,
    bures_distance,
)
from qot_course.quantum.composite import bell_state, tensor
from qot_course.quantum.density import (
    density_matrix, maximally_mixed, von_neumann_entropy,
)
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS

np.random.seed(42)
viz.use_course_style()

## 1. Recap — von Neumann entropy

Quick refresher from S3: the **von Neumann entropy** $S(\rho) = -\mathrm{tr}(\rho\log\rho)$
generalises Shannon entropy. It is $0$ for a pure state, maximal ($\log_2 d$ bits) for the
maximally mixed state $I/d$.

In [ ]:
for name, state in [("|0⟩", KET_0), ("|+⟩", KET_PLUS)]:
    print(f"S({name}⟨{name[1:-1]}|) = {von_neumann_entropy(density_matrix(state)):.3f} bit  (pure)")
print(f"S(I/2)        = {von_neumann_entropy(maximally_mixed(2)):.3f} bit  (maximally mixed)")

**Read the output.** Pure states have $S = 0$ (perfect certainty about the state);
$I/2$ has $S = 1$ bit (a coin flip, one bit of uncertainty). This is the *quantum*
notion of uncertainty; we now build the quantum versions of KL, MI, and CMI on top.

## 2. Umegaki relative entropy — the quantum KL

The quantum analogue of $D(p\|q)$ is

$$ S(\rho \,\|\, \sigma) = \mathrm{tr}\!\left[\rho\,(\log\rho - \log\sigma)\right]. $$

It is non-negative; zero iff $\rho = \sigma$; *asymmetric*; and **infinite** whenever
$\rho$ has support outside the support of $\sigma$ (the classical singularity persists).

### The illustrating punchline

The states $|+\rangle\langle+|$ (pure) and $I/2$ (maximally mixed) have the **same diagonal**
in the $Z$ basis — both $(1/2, 1/2)$. A classical KL between their diagonals is **zero**:
$D((1/2,1/2)\,\|\,(1/2,1/2)) = 0$. But the Umegaki relative entropy *sees the off-diagonal
coherence* and gives **1 bit**. Same measurement statistics, two genuinely different states.

In [ ]:
plus = density_matrix(KET_PLUS)
mixed = maximally_mixed(2)

print("Classical KL on the diagonals  D((1/2,1/2) || (1/2,1/2)) = 0.000 bit")
print(f"Quantum   S(|+⟩⟨+| || I/2)                       = {quantum_relative_entropy(plus, mixed):.3f} bit")
print()
print(f"Symmetry check:  S(I/2 || |+⟩⟨+|)                   = {quantum_relative_entropy(mixed, plus):>9}")
print(f"Self:            S(|+⟩⟨+| || |+⟩⟨+|)                   = {quantum_relative_entropy(plus, plus):.3f}")
print()
print(f"Disjoint support: S(|0⟩⟨0| || |1⟩⟨1|) = {quantum_relative_entropy(density_matrix(KET_0), density_matrix(KET_1)):>9}")

**Read the output.**
- $S(|+\rangle\langle+|\,\|\,I/2) = 1$ bit: the *coherences* matter. Classical KL was blind to them.
- The relative entropy is *asymmetric*: $S(I/2\,\|\,|+\rangle\langle+|) = \infty$ because $I/2$ has support outside the rank-1 projection $|+\rangle\langle+|$. KL "going the other way" is the regime that becomes infinite.
- Disjoint supports give $\infty$ — same singularity as classical KL when supports do not align.

The Umegaki relative entropy is the **parent quantity** of all of M4: entropic-regularised quantum optimal transport (S14) is a quantum KL projection. We will revisit this in two months.

## 3. Quantum mutual information — twice the classical bound

By analogy with $I(X;Y) = D(p_{XY}\,\|\,p_X p_Y)$, the **quantum mutual information** is

$$ I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB}). $$

For a **product** state, $I(A{:}B) = 0$. For a **Bell** state, watch closely:
$\rho_{AB}$ is *pure* (entropy 0), but each reduced state is maximally mixed (entropy 1
each). So $I(A{:}B) = 1 + 1 - 0 = \mathbf{2}$ bits — **double the classical bound**.
This is impossible classically: classical mutual information $I(X;Y) \le \min(H(X), H(Y))$.

In [ ]:
rho_bell    = density_matrix(bell_state())
rho_product = density_matrix(tensor(KET_0, KET_PLUS))

print(f"I(A:B)  Bell state    = {quantum_mutual_information(rho_bell,    dims=[2, 2]):.3f} bit")
print(f"I(A:B)  product state = {quantum_mutual_information(rho_product, dims=[2, 2]):.3f} bit")
print()
print("Classical bound (any joint distribution of two bits):  I(X;Y) ≤ 1 bit.")
print("Quantum Bell pair beats it by a factor of 2 → entanglement signature.")

**Read the output.** Two bits of mutual information between two qubits, each individually
maximally uncertain. The whole knows what the parts cannot. We will *quantify* this
gap below by the Werner-state sweep.

## 4. Negative conditional entropy — *the* signature of entanglement

The quantum conditional entropy is

$$ S(A|B) = S(\rho_{AB}) - S(\rho_B). $$

Classically $H(X|Y) \ge 0$: knowing $Y$ can only reduce uncertainty about $X$, not make it
*negative*. Quantum mechanics breaks this. For a Bell pair: $S(\rho_{AB}) = 0$ (pure),
$S(\rho_B) = 1$ bit (maximally mixed reduced state). So **$S(A|B) = -1$ bit** (Cerf & Adami, 1997).

Operationally, this negative number is the **coherent information** — the rate at which
one can teleport quantum information *back* to Alice using the shared entanglement.

In [ ]:
print(f"S(A|B)  Bell state    = {quantum_conditional_entropy(rho_bell,    dims=[2, 2]):+.3f} bit  (impossible classically)")
print(f"S(A|B)  product state = {quantum_conditional_entropy(rho_product, dims=[2, 2]):+.3f} bit")
print()
print("Classical bound:  H(X|Y) ≥ 0  ALWAYS.")
print("Quantum entanglement violates it.")

### Visualise it — the Werner-state sweep

We mix the Bell state with white noise:
$$ \rho(p) = (1 - p)\, |\mathrm{Bell}\rangle\langle\mathrm{Bell}| + p\, \frac{I_4}{4}, \qquad p \in [0, 1]. $$
At $p = 0$ we have the pure Bell pair; at $p = 1$ we have the maximally mixed two-qubit
state. We plot $I(A{:}B)$ and $S(A|B)$ across the sweep.

In [ ]:
def werner_state(p):
    return (1.0 - p) * density_matrix(bell_state()) + p * (np.eye(4, dtype=complex) / 4.0)

ps = np.linspace(0.0, 1.0, 60)
qmi = np.array([quantum_mutual_information(werner_state(p), dims=[2, 2]) for p in ps])
qce = np.array([quantum_conditional_entropy(werner_state(p), dims=[2, 2]) for p in ps])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

axes[0].plot(ps, qmi, color=viz.SOURCE_COLOR, lw=2)
axes[0].axhline(1.0, color=viz.TARGET_COLOR, linestyle="--", label="classical max (1 bit)")
axes[0].axhline(0.0, color="#94a3b8", linewidth=0.8)
axes[0].set_xlabel("Werner noise parameter  $p$")
axes[0].set_ylabel("$I(A{:}B)$  (bits)")
axes[0].set_title("Quantum mutual information", pad=10)
axes[0].legend(loc="upper right")

axes[1].plot(ps, qce, color=viz.FLOW_COLOR, lw=2)
axes[1].axhline(0.0, color=viz.TARGET_COLOR, linestyle="--", label="classical floor (0)")
axes[1].fill_between(ps, qce, 0.0, where=qce < 0.0, color=viz.FLOW_COLOR, alpha=0.18,
                     label="negative region (entangled)")
axes[1].set_xlabel("Werner noise parameter  $p$")
axes[1].set_ylabel("$S(A|B)$  (bits)")
axes[1].set_title("Quantum conditional entropy", pad=10)
axes[1].legend(loc="lower right")

fig.suptitle("Werner-state sweep: two purely-quantum effects", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Read both panels.**
- **Left.** $I(A{:}B)$ starts at **2 bits** (pure Bell, double the classical maximum) and
  decays smoothly to **0** as the state becomes the maximally mixed $I/4$. The dashed
  amber line marks the classical maximum (1 bit) — *everything above it is quantum*.
- **Right.** $S(A|B)$ starts at **$-1$ bit** (negative — impossible classically) and rises
  through zero into the positive region. The shaded green region is the **entangled
  zone**; once $S(A|B) \ge 0$, the state may be classically correlated.

Two purely-quantum effects, one sweep, shared origin: **entanglement**. As $p$ grows, the
shared resource is destroyed and both quantities relax to their classical regime.

*Caveat.* $S(A|B) \ge 0$ is *necessary* but not *sufficient* for separability — there are
"bound-entangled" states with non-negative conditional entropy. Entanglement is harder
to detect than to break.

## 5. Bures distance — the quantum lift of Fisher–Rao

The S6 Fisher–Rao distance becomes the **Bures distance** in the quantum setting:

$$ d_B(\rho, \sigma) = \sqrt{2\,(1 - F_U(\rho, \sigma))}, \qquad
F_U(\rho, \sigma) = \mathrm{tr}\sqrt{\sqrt{\rho}\,\sigma\sqrt{\rho}}\ \ (\text{Uhlmann fidelity}). $$

For *diagonal* states ($\rho = \mathrm{diag}(p)$, $\sigma = \mathrm{diag}(q)$),
$F_U = \sum_i \sqrt{p_i\, q_i}$ — the **Bhattacharyya coefficient** of S6 — and Bures
becomes the Hellinger distance, the small-angle linearisation of Fisher–Rao.
For *non-commuting* states the metric goes beyond what the simplex can see.

In [ ]:
pairs = [
    ("|+⟩ vs |+⟩",                density_matrix(KET_PLUS), density_matrix(KET_PLUS)),
    ("|0⟩ vs |1⟩   (orthogonal)", density_matrix(KET_0),    density_matrix(KET_1)),
    ("|+⟩ vs |-⟩   (orthogonal)", density_matrix(KET_PLUS), density_matrix(KET_MINUS)),
    ("|+⟩ vs I/2  (same diagonal!)",   density_matrix(KET_PLUS), maximally_mixed(2)),
    ("|0⟩ vs I/2",                      density_matrix(KET_0),    maximally_mixed(2)),
]
for label, rho, sigma in pairs:
    print(f"d_B({label:<32s}) = {bures_distance(rho, sigma):.4f}")

**Read the output.**
- Identical states are at distance $0$.
- Orthogonal pure states are at the maximum $\sqrt{2} \approx 1.414$.
- $|+\rangle$ vs $I/2$ has the **same $Z$-diagonal** as the previous self-comparison, yet
  $d_B$ is strictly positive — Bures *sees* the coherence. This is the same effect as
  Umegaki separating them above.
- *Quantum Fisher information* (QFI) is the infinitesimal version of $d_B^2$. For a
  one-parameter family $\rho(\theta)$, $\mathrm{QFI}(\theta) = 4\,(\partial_\theta\sqrt{F_U})^2$
  in the simplest case; for a pure-state rotation on the Bloch sphere it is constantly $1$
  — saturating the classical Fisher in any measurement basis. (Wilde, ch. 12.)

## 6. Dictionary — information row completed

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| **KL divergence $D(p\|q)$** | **Umegaki relative entropy $S(\rho\|\sigma)$** |
| **mutual information $I(X;Y)$** | **quantum MI $I(A{:}B)$** (can be twice the classical max) |
| **conditional entropy $H(X|Y) \ge 0$** | **quantum CMI $S(A|B)$** — *can be negative* |
| **Fisher–Rao metric** | **Bures metric / quantum Fisher information** |
| Markov kernel | quantum channel (CPTP) |
| independent variables | product state |

**M2 (the spine) is done.** From S8 onward we leave the information side and pick up
the *transport* side — the second geometry of the simplex we met in S6.

## 7. Your turn

1. **Werner separability threshold.** Find numerically the smallest $p$ at which
   $S(A|B) \ge 0$ for the Werner state. Compare to the **Peres–Horodecki PPT bound**
   $p \ge 2/3$ for separability — does negative-CMI track separability exactly, or is
   it a strict sub-criterion? *(Hint: it is strict — there exist states with $S(A|B) \ge 0$
   that are still entangled.)*
2. **Asymmetry of Umegaki.** Pick two *mixed* qubit states $\rho, \sigma$ (e.g. two
   different Bloch vectors), compute $S(\rho\|\sigma)$ and $S(\sigma\|\rho)$, and check
   they differ. Compute their *symmetric* average — does it satisfy the triangle
   inequality on a few examples?
3. **Bures along a Bloch rotation.** For the family
   $|\psi(\theta)\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$,
   tabulate $d_B(|\psi(0)\rangle, |\psi(\theta)\rangle)$ for $\theta \in \{0, \pi/4,
   \pi/2, \pi\}$ and confirm it equals $2\sin(\theta/4)$.

## Conclusions & references

- **Umegaki relative entropy** $S(\rho\|\sigma)$ is the quantum KL — the parent of every
  entropic regulariser in QOT (Sinkhorn, S10; quantum Sinkhorn, S14).
- **Quantum mutual information** can double the classical bound; **conditional entropy**
  can go negative. Both are entanglement signatures with no classical counterpart.
- **Bures distance** is the quantum lift of Fisher–Rao; for diagonal states it reduces to
  Hellinger / Bhattacharyya.
- **M2 (the spine) is complete.** Next (**S8 — Monge → Kantorovich**) we leave information
  for *transport*: how to compute the cheapest way to move one pile of mass into another.

**References:** M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum
Information* (2010), ch. 11; M. M. Wilde, *Quantum Information Theory* (2017), chs. 11–12;
N. J. Cerf & C. Adami, "Negative entropy and information in quantum mechanics",
Phys. Rev. Lett. 79, 5194 (1997); A. Uhlmann, Rep. Math. Phys. 9, 273 (1976);
H. Umegaki, "Conditional expectation in an operator algebra. IV. Entropy and information",
Kodai Math. Sem. Rep. 14, 59 (1962). Previous: `notebooks/02_InformationAndGeometry/02_geometry.ipynb`.